# Chronos-2 Inference + Forking-Sequences-Style Ensembling (Toy Demo)

This notebook demonstrates the zero-shot forecast ensembling applied to
pretrained time series foundation models (**Chronos-2** and **TimesFM 2.5**).

**What we'll do:**
1. Wrap Chronos-2 (and later TimesFM 2.5) behind a small inference class
2. Generate a toy synthetic series with trend + seasonality + noise
3. Run **rolling inference**: forecast from many overlapping forecast creation
   dates (FCDs), producing a grid of overlapping multi-horizon forecasts
   — the same grid structure forking-sequences produces natively
4. Apply the `Ensembler` class to combine overlapping forecasts for each target date
5. Visualize forecast volatility **before vs. after** ensembling

> **Note on environment:** this notebook calls the real Chronos-2 and
> TimesFM 2.5 checkpoints directly — `chronos-forecasting` / `timesfm`
> and internet access to Hugging Face are required. There is no mock
> fallback, so if either package or checkpoint is unavailable, the
> corresponding wrapper's `__init__` will raise.


## Setup

Imports, RNG seeds, and the `sys.path` hookup needed to pull `Ensembler` and
the stability/loss metrics straight out of the `tsforge` package.

In [ ]:
# If needed:
# !pip install chronos-forecasting torch numpy matplotlib pandas timesfm[torch]

import os
import sys

import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(os.path.join("..", "tsforge")))
from common.ensembling import Ensembler
from metrics.stability_metrics import forecast_percentage_change, excess_volatility
from metrics.eval_losses import crps

np.random.seed(0)
torch.manual_seed(0)


In [ ]:
def make_toy_series(n=200, trend=0.03, seasonal_amp=8.0, period=24, noise_std=2.0):
    t = np.arange(n)
    series = 50 + trend * t + seasonal_amp * np.sin(2 * np.pi * t / period)
    series += np.random.normal(0, noise_std, size=n)
    return series

series = make_toy_series()

plt.figure(figsize=(10, 3))
plt.plot(series)
plt.title("Toy series: trend + seasonality + noise")
plt.xlabel("t")
plt.ylabel("value")
plt.tight_layout()
plt.show()


## Chronos-2 Wrapper

A wrapper around `chronos.BaseChronosPipeline` that returns quantile
forecasts given a context window. Requires `chronos-forecasting` and
internet access to Hugging Face — no mock fallback.


In [ ]:

class ChronosWrapper:
    '''Thin wrapper around Chronos-2 for quantile forecasting.'''

    def __init__(self, model_id="amazon/chronos-2", device_map="cpu", torch_dtype=torch.float32):
        from chronos import BaseChronosPipeline
        self.pipeline = BaseChronosPipeline.from_pretrained(
            model_id, device_map=device_map, torch_dtype=torch_dtype
        )
        print(f"[ChronosWrapper] Loaded real model: {model_id}")

    def predict_quantiles(self, context, prediction_length, quantile_levels=(0.1, 0.5, 0.9)):
        '''
        context: 1D array-like of past values
        returns:
            quantiles: np.ndarray [len(quantile_levels), prediction_length]
            mean:      np.ndarray [prediction_length]
        '''
        # Chronos2Pipeline expects a 3-d tensor: (n_series, n_variates, history_length)
        context_t = torch.tensor(np.asarray(context), dtype=torch.float32).reshape(1, 1, -1)
        quantiles, mean = self.pipeline.predict_quantiles(
            inputs=context_t,
            prediction_length=prediction_length,
            quantile_levels=list(quantile_levels),
        )
        # quantiles[0]: (n_variates=1, H, Q) -> (H, Q) -> (Q, H)
        q = quantiles[0].squeeze(0).numpy().T
        m = mean[0].squeeze(0).numpy()
        return q, m



## Rolling Inference Across FCDs

Window-sampling generates one forecast per forward pass. To build the same
kind of *overlapping forecast grid* that forking-sequences produces natively
in a single pass, we simulate it here with a rolling loop: at each forecast
creation date (FCD), we feed Chronos-2 the history up to that point and get
back an `H`-step-ahead quantile forecast. Stacking these across many FCDs
gives overlapping predictions for the same target dates.


In [ ]:

model = ChronosWrapper(model_id="amazon/chronos-2")

context_len = 64     # minimum history required before the first forecast
H = 12                # forecast horizon
Q_LEVELS = (0.1, 0.5, 0.9)
stride = 1             # roll forward by 1 step at each FCD

fcds = list(range(context_len, len(series) - H, stride))
T = len(fcds)
C = 1  # single channel/series
Q = len(Q_LEVELS)

preds = np.full((1, T, H, C, Q), np.nan)
targets = np.full((1, T, H, C), np.nan)    
for i, fcd in enumerate(fcds):
    context = series[:fcd]
    quantiles, _ = model.predict_quantiles(context, prediction_length=H, quantile_levels=Q_LEVELS)
    # quantiles: [Q, H] -> store as [H, C=1, Q]
    preds[0, i, :, 0, :] = quantiles.T
    targets[0, i, :, 0] = series[fcd:fcd + H]

mask_t = np.ones((1, T, H, C))

print("preds shape (B, T, H, C, Q):", tuple(preds.shape))


## Forecast Ensembling - Exponentially Weighted Moving Average (EWM)

Combines overlapping forecasts for each target date with exponential
smoothing (α = 0.9), weighting the most recent (shortest-horizon, most
accurate) revision most heavily.

In [ ]:
ensembler = Ensembler(ensemble_method='ewm', stride=stride, alpha=0.9)
ensembled_preds = ensembler.ensemble(preds, mask_t)  # (B, T, H, C, Q)

P10_IDX = Q_LEVELS.index(0.1)
P50_IDX = Q_LEVELS.index(0.5)
P90_IDX = Q_LEVELS.index(0.9)

n_show = 24
show_idx = np.linspace(0, T - 1, n_show, dtype=int)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'serif'
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

for ax, title, grid in zip(axes, ["Original (no ensembling)", "Ensembled (EWM α=0.9)"], [preds, ensembled_preds]):
    ax.plot(series, color='black', lw=1, label='True series')
    for j, i in enumerate(show_idx):
        fcd = fcds[i]
        xs = np.arange(fcd, fcd + H)
        p10 = grid[0, i, :, 0, P10_IDX]
        p50 = grid[0, i, :, 0, P50_IDX]
        p90 = grid[0, i, :, 0, P90_IDX]
        color = plt.cm.viridis(j / max(n_show - 1, 1))
        ax.plot(xs, p50, color=color, lw=1.3)
        ax.fill_between(xs, p10, p90, color=color, alpha=0.2)
        ax.axvline(fcd, color=color, ls=':', lw=0.6)
    ax.set_title(title)
    ax.set_xlabel('t')

axes[0].set_ylabel('value')
axes[0].legend(fontsize=7, loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
quantiles = [0.1, 0.5, 0.9]

scrps_before = crps(preds=preds, targets=targets, quantiles=quantiles, mask=mask_t)
scrps_after = crps(preds=ensembled_preds, targets=targets, quantiles=quantiles, mask=mask_t)

print(f"sCRPS (no ensembling):     {scrps_before:.3f}")
print(f"sCPRS (ensembled, α=0.9):  {scrps_after:.3f}")
print(f"Relative reduction:       {100 * (scrps_before - scrps_after) / scrps_before:.1f}%")
print(' ')

sfpc_before = forecast_percentage_change(preds=preds[:, :, :, :, 1], stride=1, symmetric=True, mask=mask_t)
sfpc_after = forecast_percentage_change(preds=ensembled_preds[:, :, :, :, 1], stride=1, symmetric=True, mask=mask_t)

print(f"sFPC (no ensembling):     {sfpc_before:.3f}")
print(f"sFPC (ensembled, α=0.9):  {sfpc_after:.3f}")
print(f"Relative reduction:       {100 * (sfpc_before - sfpc_after) / sfpc_before:.1f}%")
print(' ')

sev_before = excess_volatility(preds=preds, targets=targets, quantiles=quantiles, stride=1, scaling=True, mask=mask_t)
sev_after = excess_volatility(preds=ensembled_preds, targets=targets, quantiles=quantiles, stride=1, scaling=True, mask=mask_t)

print(f"sEV (no ensembling):     {sev_before:.3f}")
print(f"sEV (ensembled, α=0.9):  {sev_after:.3f}")
if sfpc_before > 0:
    print(f"Relative reduction:       {100 * (sev_before - sev_after) / sfpc_before:.1f}%")


## Forecast Ensembling - Mean

Combines overlapping forecasts for each target date with a simple,
unweighted average, as a baseline to compare against EWM.

In [ ]:
ensembler = Ensembler(ensemble_method='mean', stride=stride, alpha=0.9)
ensembled_preds = ensembler.ensemble(preds, mask_t)  # (B, T, H, C, Q)

P10_IDX = Q_LEVELS.index(0.1)
P50_IDX = Q_LEVELS.index(0.5)
P90_IDX = Q_LEVELS.index(0.9)

n_show = 24
show_idx = np.linspace(0, T - 1, n_show, dtype=int)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'serif'
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, title, grid in zip(axes, ["Original (no ensembling)", "Ensembled (mean)"], [preds, ensembled_preds]):
    ax.plot(series, color='black', lw=1, label='True series')
    for j, i in enumerate(show_idx):
        fcd = fcds[i]
        xs = np.arange(fcd, fcd + H)
        p10 = grid[0, i, :, 0, P10_IDX]
        p50 = grid[0, i, :, 0, P50_IDX]
        p90 = grid[0, i, :, 0, P90_IDX]
        color = plt.cm.viridis(j / max(n_show - 1, 1))
        ax.plot(xs, p50, color=color, lw=1.3) #, label=f'FCD={fcd}')
        ax.fill_between(xs, p10, p90, color=color, alpha=0.2)
        ax.axvline(fcd, color=color, ls=':', lw=0.6)
    ax.set_title(title)
    ax.set_xlabel('t')

axes[0].set_ylabel('value')
axes[0].legend(fontsize=7, loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
quantiles = [0.1, 0.5, 0.9]

scrps_before = crps(preds=preds, targets=targets, quantiles=quantiles, mask=mask_t)
scrps_after = crps(preds=ensembled_preds, targets=targets, quantiles=quantiles, mask=mask_t)

print(f"sCRPS (no ensembling):     {scrps_before:.3f}")
print(f"sCPRS (ensembled, α=0.9):  {scrps_after:.3f}")
print(f"Relative reduction:       {100 * (scrps_before - scrps_after) / scrps_before:.1f}%")
print(' ')

sfpc_before = forecast_percentage_change(preds=preds[:, :, :, :, 1], stride=1, symmetric=True, mask=mask_t)
sfpc_after = forecast_percentage_change(preds=ensembled_preds[:, :, :, :, 1], stride=1, symmetric=True, mask=mask_t)

print(f"sFPC (no ensembling):     {sfpc_before:.3f}")
print(f"sFPC (ensembled, α=0.9):  {sfpc_after:.3f}")
print(f"Relative reduction:       {100 * (sfpc_before - sfpc_after) / sfpc_before:.1f}%")
print(' ')

sev_before = excess_volatility(preds=preds, targets=targets, quantiles=quantiles, stride=1, scaling=True, mask=mask_t)
sev_after = excess_volatility(preds=ensembled_preds, targets=targets, quantiles=quantiles, stride=1, scaling=True, mask=mask_t)

print(f"sEV (no ensembling):     {sev_before:.3f}")
print(f"sEV (ensembled, α=0.9):  {sev_after:.3f}")
if sfpc_before > 0:
    print(f"Relative reduction:       {100 * (sev_before - sev_after) / sfpc_before:.1f}%")


## TimesFM 2.5 Wrapper

Same idea as `ChronosWrapper`, now for Google's **TimesFM 2.5**
(`google/timesfm-2.5-200m-pytorch`), behind the identical
`predict_quantiles(context, prediction_length, quantile_levels)` interface.

In [ ]:
class TimesFMWrapper:
    '''Thin wrapper around TimesFM 2.5 for quantile forecasting.'''

    def __init__(self, model_id="google/timesfm-2.5-200m-pytorch", max_context=512, max_horizon=128):
        import timesfm
        self.model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(model_id)
        self.model.compile(
            timesfm.ForecastConfig(
                max_context=max_context,
                max_horizon=max_horizon,
                normalize_inputs=True,
                use_continuous_quantile_head=True,
                fix_quantile_crossing=True,
            )
        )
        print(f"[TimesFMWrapper] Loaded real model: {model_id}")

    def predict_quantiles(self, context, prediction_length, quantile_levels=(0.1, 0.5, 0.9)):
        '''
        context: 1D array-like of past values
        returns:
            quantiles: np.ndarray [len(quantile_levels), prediction_length]
            mean:      np.ndarray [prediction_length]
        '''
        _, quantile_forecast = self.model.forecast(
            horizon=prediction_length, inputs=[np.asarray(context, dtype=float)]
        )
        # quantile_forecast[0]: [H, 10] -> col 0 is the mean, cols 1..9 are P10..P90
        qf = np.asarray(quantile_forecast[0])
        col = {round(0.1 * k, 1): k for k in range(1, 10)}
        quantiles = np.stack([qf[:, col[round(q, 1)]] for q in quantile_levels], axis=0)
        mean = qf[:, 0]
        return quantiles, mean


## Rolling Inference Across FCDs (TimesFM 2.5)

In [ ]:
model_tfm = TimesFMWrapper(model_id="google/timesfm-2.5-200m-pytorch")

context_len = 64     # minimum history required before the first forecast
H = 12                # forecast horizon
Q_LEVELS = (0.1, 0.5, 0.9)
stride = 1             # roll forward by 1 step at each FCD

fcds = list(range(context_len, len(series) - H, stride))
T = len(fcds)
C = 1  # single channel/series
Q = len(Q_LEVELS)

mask_t = np.ones((1, T, H, C))

preds_tfm = np.full((1, T, H, C, Q), np.nan)
targets = np.full((1, T, H, C), np.nan)
for i, fcd in enumerate(fcds):
    context = series[:fcd]
    quantiles, _ = model_tfm.predict_quantiles(context, prediction_length=H, quantile_levels=Q_LEVELS)
    preds_tfm[0, i, :, 0, :] = quantiles.T
    targets[0, i, :, 0] = series[fcd:fcd + H]

print("preds_tfm shape (B, T, H, C, Q):", tuple(preds_tfm.shape))


## Forecast Ensembling - EWM

In [ ]:
ensembler = Ensembler(ensemble_method='ewm', stride=stride, alpha=0.9)
ensembled_preds_tfm = ensembler.ensemble(preds_tfm, mask_t)  # (B, T, H, C, Q)

P10_IDX = Q_LEVELS.index(0.1)
P50_IDX = Q_LEVELS.index(0.5)
P90_IDX = Q_LEVELS.index(0.9)

n_show = 24
show_idx = np.linspace(0, T - 1, n_show, dtype=int)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'serif'
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

for ax, title, grid in zip(axes, ["Original (no ensembling)", "Ensembled (EWM α=0.9)"], [preds_tfm, ensembled_preds_tfm]):
    ax.plot(series, color='black', lw=1, label='True series')
    for j, i in enumerate(show_idx):
        fcd = fcds[i]
        xs = np.arange(fcd, fcd + H)
        p10 = grid[0, i, :, 0, P10_IDX]
        p50 = grid[0, i, :, 0, P50_IDX]
        p90 = grid[0, i, :, 0, P90_IDX]
        color = plt.cm.viridis(j / max(n_show - 1, 1))
        ax.plot(xs, p50, color=color, lw=1.3)
        ax.fill_between(xs, p10, p90, color=color, alpha=0.2)
        ax.axvline(fcd, color=color, ls=':', lw=0.6)
    ax.set_title(f"TimesFM 2.5 - {title}")
    ax.set_xlabel('t')

axes[0].set_ylabel('value')
axes[0].legend(fontsize=7, loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
quantiles = [0.1, 0.5, 0.9]

scrps_before = crps(preds=preds_tfm, targets=targets, quantiles=quantiles, mask=mask_t)
scrps_after = crps(preds=ensembled_preds_tfm, targets=targets, quantiles=quantiles, mask=mask_t)

print(f"sCRPS (no ensembling):     {scrps_before:.3f}")
print(f"sCPRS (ensembled, α=0.9):  {scrps_after:.3f}")
print(f"Relative reduction:       {100 * (scrps_before - scrps_after) / scrps_before:.1f}%")
print(' ')

sfpc_before = forecast_percentage_change(preds=preds_tfm[:, :, :, :, 1], stride=1, symmetric=True, mask=mask_t)
sfpc_after = forecast_percentage_change(preds=ensembled_preds_tfm[:, :, :, :, 1], stride=1, symmetric=True, mask=mask_t)

print(f"sFPC (no ensembling):     {sfpc_before:.3f}")
print(f"sFPC (ensembled, α=0.9):  {sfpc_after:.3f}")
print(f"Relative reduction:       {100 * (sfpc_before - sfpc_after) / sfpc_before:.1f}%")
print(' ')

sev_before = excess_volatility(preds=preds_tfm, targets=targets, quantiles=quantiles, stride=1, scaling=True, mask=mask_t)
sev_after = excess_volatility(preds=ensembled_preds_tfm, targets=targets, quantiles=quantiles, stride=1, scaling=True, mask=mask_t)

print(f"sEV (no ensembling):     {sev_before:.3f}")
print(f"sEV (ensembled, α=0.9):  {sev_after:.3f}")
if sfpc_before > 0:
    print(f"Relative reduction:       {100 * (sev_before - sev_after) / sfpc_before:.1f}%")


## Forecast Ensembling - Mean

In [ ]:
ensembler = Ensembler(ensemble_method='mean', stride=stride, alpha=0.9)
ensembled_preds_tfm = ensembler.ensemble(preds_tfm, mask_t)  # (B, T, H, C, Q)

n_show = 24
show_idx = np.linspace(0, T - 1, n_show, dtype=int)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'serif'
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, title, grid in zip(axes, ["Original (no ensembling)", "Ensembled (mean)"], [preds_tfm, ensembled_preds_tfm]):
    ax.plot(series, color='black', lw=1, label='True series')
    for j, i in enumerate(show_idx):
        fcd = fcds[i]
        xs = np.arange(fcd, fcd + H)
        p10 = grid[0, i, :, 0, P10_IDX]
        p50 = grid[0, i, :, 0, P50_IDX]
        p90 = grid[0, i, :, 0, P90_IDX]
        color = plt.cm.viridis(j / max(n_show - 1, 1))
        ax.plot(xs, p50, color=color, lw=1.3)
        ax.fill_between(xs, p10, p90, color=color, alpha=0.2)
        ax.axvline(fcd, color=color, ls=':', lw=0.6)
    ax.set_title(f"TimesFM 2.5 - {title}")
    ax.set_xlabel('t')

axes[0].set_ylabel('value')
axes[0].legend(fontsize=7, loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
scrps_before = crps(preds=preds_tfm, targets=targets, quantiles=quantiles, mask=mask_t)
scrps_after = crps(preds=ensembled_preds_tfm, targets=targets, quantiles=quantiles, mask=mask_t)

print(f"sCRPS (no ensembling):     {scrps_before:.3f}")
print(f"sCPRS (ensembled, mean):  {scrps_after:.3f}")
print(f"Relative reduction:       {100 * (scrps_before - scrps_after) / scrps_before:.1f}%")
print(' ')

sfpc_before = forecast_percentage_change(preds=preds_tfm[:, :, :, :, 1], stride=1, symmetric=True, mask=mask_t)
sfpc_after = forecast_percentage_change(preds=ensembled_preds_tfm[:, :, :, :, 1], stride=1, symmetric=True, mask=mask_t)

print(f"sFPC (no ensembling):     {sfpc_before:.3f}")
print(f"sFPC (ensembled, mean):  {sfpc_after:.3f}")
print(f"Relative reduction:       {100 * (sfpc_before - sfpc_after) / sfpc_before:.1f}%")
print(' ')

sev_before = excess_volatility(preds=preds_tfm, targets=targets, quantiles=quantiles, stride=1, scaling=True, mask=mask_t)
sev_after = excess_volatility(preds=ensembled_preds_tfm, targets=targets, quantiles=quantiles, stride=1, scaling=True, mask=mask_t)

print(f"sEV (no ensembling):     {sev_before:.3f}")
print(f"sEV (ensembled, mean):  {sev_after:.3f}")
if sfpc_before > 0:
    print(f"Relative reduction:       {100 * (sev_before - sev_after) / sfpc_before:.1f}%")
